# Predicting Heart Disease - S6E2
LightGBM + XGBoost + CatBoost × Multi-Seed Averaging

**v3: Multi-Seed版**

In [9]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from scipy.stats import rankdata
import warnings
warnings.filterwarnings('ignore')

In [10]:
class CFG:
    n_splits = 5
    target_col = 'Heart Disease'
    use_original = True
    # Multi-Seed: 各モデルをこれらのseedで繰り返し学習して平均化
    seeds = [42, 123, 456, 789, 2024]

class Paths:
    p = '/Users/shirokoshikentaro/Desktop/Predicting Heart Disease/Data'
    train = p + '/train.csv'
    test = p + '/test.csv'
    sample = p + '/sample_submission.csv'
    original = p + '/dataset_heart.csv'

## 1. Data Loading + 原データ追加

In [11]:
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)
sample_submission = pd.read_csv(Paths.sample)

print(f'Train shape (synthetic only): {train.shape}')
print(f'Test shape: {test.shape}')

if CFG.use_original:
    original = pd.read_csv(Paths.original)
    print(f'Original shape: {original.shape}')
    print(f'Original columns: {original.columns.tolist()}')

    # カラム名を完全にリネーム（原データ → コンペデータ）
    rename_map = {
        'age': 'Age',
        'sex ': 'Sex',                                    # 末尾にスペースあり注意
        'chest pain type': 'Chest pain type',
        'resting blood pressure': 'BP',
        'serum cholestoral': 'Cholesterol',
        'fasting blood sugar': 'FBS over 120',
        'resting electrocardiographic results': 'EKG results',
        'max heart rate': 'Max HR',
        'exercise induced angina': 'Exercise angina',
        'oldpeak': 'ST depression',
        'ST segment': 'Slope of ST',
        'major vessels': 'Number of vessels fluro',
        'thal': 'Thallium',
        'heart disease': 'Heart Disease',
    }
    original = original.rename(columns=rename_map)

    # ターゲット変換（Statlog形式: 1=Absence→0, 2=Presence→1）
    original['Heart Disease'] = original['Heart Disease'].map({1: 0, 2: 1})

    print(f'\nAfter rename columns: {original.columns.tolist()}')
    print(f'Target unique: {original["Heart Disease"].unique()}')
    print(f'Target distribution:\n{original["Heart Disease"].value_counts()}')
    print(f'NaN in target: {original["Heart Disease"].isnull().sum()}')

Train shape (synthetic only): (630000, 15)
Test shape: (270000, 14)
Original shape: (270, 14)
Original columns: ['age', 'sex ', 'chest pain type', 'resting blood pressure', 'serum cholestoral', 'fasting blood sugar', 'resting electrocardiographic results', 'max heart rate', 'exercise induced angina', 'oldpeak', 'ST segment', 'major vessels', 'thal', 'heart disease']

After rename columns: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium', 'Heart Disease']
Target unique: [1 0]
Target distribution:
Heart Disease
0    150
1    120
Name: count, dtype: int64
NaN in target: 0


## 2. Target Encoding

In [12]:
if train[CFG.target_col].dtype == 'object' or train[CFG.target_col].dtype.name == 'category':
    train[CFG.target_col] = train[CFG.target_col].map({
        'Absence': 0, 'Presence': 1,
        'No': 0, 'Yes': 1,
        0: 0, 1: 1,
    })

print(f'Target dtype: {train[CFG.target_col].dtype}')
print(f'Target unique: {train[CFG.target_col].unique()}')
print(f'Target distribution:\n{train[CFG.target_col].value_counts(normalize=True)}')
assert train[CFG.target_col].isnull().sum() == 0, 'ERROR: NaN in target!'

Target dtype: int64
Target unique: [1 0]
Target distribution:
Heart Disease
0    0.55166
1    0.44834
Name: proportion, dtype: float64


## 3. Feature Engineering

In [13]:
def add_features(df):
    for col in df.select_dtypes(include=['int8', 'int16']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float16']).columns:
        df[col] = df[col].astype('float32')

    df['Age_x_MaxHR'] = df['Age'] * df['Max HR']
    df['Age_div_MaxHR'] = df['Age'] / (df['Max HR'] + 1e-5)
    df['STdep_x_MaxHR'] = df['ST depression'] * df['Max HR']
    df['BP_x_Chol'] = df['BP'] * df['Cholesterol']
    df['Age_x_STdep'] = df['Age'] * df['ST depression']
    df['Age_x_Vessels'] = df['Age'] * df['Number of vessels fluro']
    df['BP_x_MaxHR'] = df['BP'] * df['Max HR']
    df['Chol_x_Age'] = df['Cholesterol'] * df['Age']
    df['STdep_x_Vessels'] = df['ST depression'] * df['Number of vessels fluro']

    df['Sex_x_MaxHR'] = df['Sex'] * df['Max HR']
    df['Sex_x_Chol'] = df['Sex'] * df['Cholesterol']
    df['ChestPain_x_STdep'] = df['Chest pain type'] * df['ST depression']
    df['ChestPain_x_MaxHR'] = df['Chest pain type'] * df['Max HR']
    df['ExAngina_x_STdep'] = df['Exercise angina'] * df['ST depression']
    df['ExAngina_x_MaxHR'] = df['Exercise angina'] * df['Max HR']
    df['Thallium_x_Vessels'] = df['Thallium'] * df['Number of vessels fluro']
    df['SlopeSTx_STdep'] = df['Slope of ST'] * df['ST depression']

    df['MaxHR_reserve'] = (220 - df['Age']) - df['Max HR']
    df['MaxHR_ratio'] = df['Max HR'] / (220 - df['Age'] + 1e-5)

    df['ChestPain_ExAngina'] = df['Chest pain type'].astype(str) + '_' + df['Exercise angina'].astype(str)
    df['ChestPain_Thallium'] = df['Chest pain type'].astype(str) + '_' + df['Thallium'].astype(str)
    df['SlopeST_ExAngina'] = df['Slope of ST'].astype(str) + '_' + df['Exercise angina'].astype(str)

    return df

train = add_features(train)
test = add_features(test)
print(f'Train shape after FE: {train.shape}')
print(f'Test shape after FE: {test.shape}')

Train shape after FE: (630000, 37)
Test shape after FE: (270000, 36)


## 4. Column Definitions

In [14]:
drop_cols = ['id', CFG.target_col]
feature_cols = [c for c in train.columns if c not in drop_cols]
cat_cols = train[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f'Feature columns ({len(feature_cols)})')
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

y_train = train[CFG.target_col]
print(f'y_train unique: {y_train.unique()}')
print(f'Seeds: {CFG.seeds} ({len(CFG.seeds)} seeds × {CFG.n_splits} folds = {len(CFG.seeds) * CFG.n_splits} models per algorithm)')

Feature columns (35)
Categorical columns (3): ['ChestPain_ExAngina', 'ChestPain_Thallium', 'SlopeST_ExAngina']
y_train unique: [1 0]
Seeds: [42, 123, 456, 789, 2024] (5 seeds × 5 folds = 25 models per algorithm)


## 5. Data Preparation (モデル別)

In [15]:
# LightGBM用
x_train_lgb = train[feature_cols].copy()
x_test_lgb = test[feature_cols].copy()
for col in cat_cols:
    x_train_lgb[col] = x_train_lgb[col].astype('category')
    x_test_lgb[col] = x_test_lgb[col].astype('category')

# XGBoost用（Label Encoding）
x_train_xgb = train[feature_cols].copy()
x_test_xgb = test[feature_cols].copy()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([x_train_xgb[col].astype(str), x_test_xgb[col].astype(str)]))
    x_train_xgb[col] = le.transform(x_train_xgb[col].astype(str))
    x_test_xgb[col] = le.transform(x_test_xgb[col].astype(str))
    label_encoders[col] = le

# CatBoost用
x_train_cat = train[feature_cols].copy()
x_test_cat = test[feature_cols].copy()
cat_col_indices = [feature_cols.index(col) for col in cat_cols]

print('Data preparation done!')

Data preparation done!


## 6. Multi-Seed Training

各seedでCV分割を変え、3モデル × 5fold × 5seed = 75モデルを学習。
同一モデルのseed間で予測を平均化することで、予測の分散を低減する。

In [16]:
# =============================================================================
# パラメータ定義
# =============================================================================
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'n_estimators': 10000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_jobs': -1,
}

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'n_estimators': 10000,
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_jobs': -1,
    'tree_method': 'hist',
    'verbosity': 0,
    'early_stopping_rounds': 100,
}

cat_params = {
    'iterations': 10000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'subsample': 0.8,
    'colsample_bylevel': 0.8,
    'eval_metric': 'AUC',
    'verbose': 0,
    'early_stopping_rounds': 100,
    'task_type': 'CPU',
}

print('Parameters defined!')

Parameters defined!


In [17]:
# =============================================================================
# Multi-Seed学習関数
# =============================================================================
def train_lgb_seed(seed):
    """LightGBMを1つのseedで5-fold学習"""
    cv = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(x_train_lgb))
    test_pred = np.zeros(len(x_test_lgb))

    params = {**lgb_params, 'random_state': seed}

    for nfold, (tr_idx, va_idx) in enumerate(cv.split(x_train_lgb, y_train)):
        x_tr, y_tr = x_train_lgb.iloc[tr_idx], y_train.iloc[tr_idx]
        x_va, y_va = x_train_lgb.iloc[va_idx], y_train.iloc[va_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            x_tr, y_tr,
            eval_set=[(x_va, y_va)],
            eval_metric='auc',
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(0),  # ログ抑制（Multi-Seedでは大量になるため）
            ],
        )
        oof[va_idx] = model.predict_proba(x_va)[:, 1]
        test_pred += model.predict_proba(x_test_lgb)[:, 1] / CFG.n_splits

    auc = roc_auc_score(y_train, oof)
    return oof, test_pred, auc


def train_xgb_seed(seed):
    """XGBoostを1つのseedで5-fold学習"""
    cv = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(x_train_xgb))
    test_pred = np.zeros(len(x_test_xgb))

    params = {**xgb_params, 'random_state': seed}

    for nfold, (tr_idx, va_idx) in enumerate(cv.split(x_train_xgb, y_train)):
        x_tr, y_tr = x_train_xgb.iloc[tr_idx], y_train.iloc[tr_idx]
        x_va, y_va = x_train_xgb.iloc[va_idx], y_train.iloc[va_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(
            x_tr, y_tr,
            eval_set=[(x_va, y_va)],
            verbose=0,  # ログ抑制
        )
        oof[va_idx] = model.predict_proba(x_va)[:, 1]
        test_pred += model.predict_proba(x_test_xgb)[:, 1] / CFG.n_splits

    auc = roc_auc_score(y_train, oof)
    return oof, test_pred, auc


def train_cat_seed(seed):
    """CatBoostを1つのseedで5-fold学習"""
    cv = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(x_train_cat))
    test_pred = np.zeros(len(x_test_cat))

    params = {**cat_params, 'random_seed': seed}

    for nfold, (tr_idx, va_idx) in enumerate(cv.split(x_train_cat, y_train)):
        x_tr, y_tr = x_train_cat.iloc[tr_idx], y_train.iloc[tr_idx]
        x_va, y_va = x_train_cat.iloc[va_idx], y_train.iloc[va_idx]

        model = CatBoostClassifier(**params)
        model.fit(
            x_tr, y_tr,
            eval_set=(x_va, y_va),
            cat_features=cat_col_indices,
        )
        oof[va_idx] = model.predict_proba(x_va)[:, 1]
        test_pred += model.predict_proba(x_test_cat)[:, 1] / CFG.n_splits

    auc = roc_auc_score(y_train, oof)
    return oof, test_pred, auc

### 6a. LightGBM × Multi-Seed

In [18]:
print('#' * 60)
print('  LightGBM × Multi-Seed')
print('#' * 60)

lgb_oofs = []
lgb_tests = []

for seed in CFG.seeds:
    oof, test_pred, auc = train_lgb_seed(seed)
    lgb_oofs.append(oof)
    lgb_tests.append(test_pred)
    print(f'  seed={seed}: OOF AUC = {auc:.6f}')

# seed間の平均
oof_lgb = np.mean(lgb_oofs, axis=0)
test_lgb = np.mean(lgb_tests, axis=0)
oof_auc_lgb = roc_auc_score(y_train, oof_lgb)
print(f'\n★ LightGBM Multi-Seed Average OOF AUC: {oof_auc_lgb:.6f}')

############################################################
  LightGBM × Multi-Seed
############################################################
  seed=42: OOF AUC = 0.955038
  seed=123: OOF AUC = 0.955005
  seed=456: OOF AUC = 0.955022
  seed=789: OOF AUC = 0.955060
  seed=2024: OOF AUC = 0.955030

★ LightGBM Multi-Seed Average OOF AUC: 0.955170


### 6b. XGBoost × Multi-Seed

In [19]:
print('#' * 60)
print('  XGBoost × Multi-Seed')
print('#' * 60)

xgb_oofs = []
xgb_tests = []

for seed in CFG.seeds:
    oof, test_pred, auc = train_xgb_seed(seed)
    xgb_oofs.append(oof)
    xgb_tests.append(test_pred)
    print(f'  seed={seed}: OOF AUC = {auc:.6f}')

oof_xgb = np.mean(xgb_oofs, axis=0)
test_xgb = np.mean(xgb_tests, axis=0)
oof_auc_xgb = roc_auc_score(y_train, oof_xgb)
print(f'\n★ XGBoost Multi-Seed Average OOF AUC: {oof_auc_xgb:.6f}')

############################################################
  XGBoost × Multi-Seed
############################################################
  seed=42: OOF AUC = 0.955036
  seed=123: OOF AUC = 0.955008
  seed=456: OOF AUC = 0.955056
  seed=789: OOF AUC = 0.955064
  seed=2024: OOF AUC = 0.955052

★ XGBoost Multi-Seed Average OOF AUC: 0.955195


### 6c. CatBoost × Multi-Seed

In [20]:
print('#' * 60)
print('  CatBoost × Multi-Seed')
print('#' * 60)

cat_oofs = []
cat_tests = []

for seed in CFG.seeds:
    oof, test_pred, auc = train_cat_seed(seed)
    cat_oofs.append(oof)
    cat_tests.append(test_pred)
    print(f'  seed={seed}: OOF AUC = {auc:.6f}')

oof_cat = np.mean(cat_oofs, axis=0)
test_cat = np.mean(cat_tests, axis=0)
oof_auc_cat = roc_auc_score(y_train, oof_cat)
print(f'\n★ CatBoost Multi-Seed Average OOF AUC: {oof_auc_cat:.6f}')

############################################################
  CatBoost × Multi-Seed
############################################################
  seed=42: OOF AUC = 0.955271
  seed=123: OOF AUC = 0.955263
  seed=456: OOF AUC = 0.955294
  seed=789: OOF AUC = 0.955297
  seed=2024: OOF AUC = 0.955292

★ CatBoost Multi-Seed Average OOF AUC: 0.955358


## 7. Ensemble: Rank Averaging

In [21]:
def rank_average(preds_list, weights=None):
    if weights is None:
        weights = [1.0 / len(preds_list)] * len(preds_list)
    ranked = [rankdata(p) / len(p) for p in preds_list]
    result = np.zeros_like(preds_list[0])
    for r, w in zip(ranked, weights):
        result += r * w
    return result

print('=== OOF AUC per model (Multi-Seed Averaged) ===')
print(f'  LightGBM : {oof_auc_lgb:.6f}')
print(f'  XGBoost  : {oof_auc_xgb:.6f}')
print(f'  CatBoost : {oof_auc_cat:.6f}')

# 等重み
oof_equal = rank_average([oof_lgb, oof_xgb, oof_cat])
auc_equal = roc_auc_score(y_train, oof_equal)
print(f'\n  Ensemble (equal) OOF AUC: {auc_equal:.6f}')

# 最適重み探索
print('\n--- Weight Optimization ---')
best_auc = 0
best_weights = None

for w1 in np.arange(0.1, 0.9, 0.05):
    for w2 in np.arange(0.1, 0.9 - w1, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0.05:
            continue
        weights = [w1, w2, w3]
        oof_blend = rank_average([oof_lgb, oof_xgb, oof_cat], weights)
        auc = roc_auc_score(y_train, oof_blend)
        if auc > best_auc:
            best_auc = auc
            best_weights = weights

print(f'  Best weights: LGB={best_weights[0]:.2f}, XGB={best_weights[1]:.2f}, CAT={best_weights[2]:.2f}')
print(f'  Best OOF AUC: {best_auc:.6f}')

test_ensemble = rank_average([test_lgb, test_xgb, test_cat], best_weights)

=== OOF AUC per model (Multi-Seed Averaged) ===
  LightGBM : 0.955170
  XGBoost  : 0.955195
  CatBoost : 0.955358

  Ensemble (equal) OOF AUC: 0.955314

--- Weight Optimization ---
  Best weights: LGB=0.10, XGB=0.10, CAT=0.80
  Best OOF AUC: 0.955364


## 8. Submission

In [22]:
submission = sample_submission.copy()
submission[CFG.target_col] = test_ensemble.astype(float)

print(f'Submission shape: {submission.shape}')
print(f'Prediction range: [{submission[CFG.target_col].min():.6f}, {submission[CFG.target_col].max():.6f}]')
print(submission.head())

submission.to_csv('submission_multiseed_v3.csv', index=False)
print('\n✅ submission_multiseed_v3.csv saved!')

Submission shape: (270000, 2)
Prediction range: [0.000007, 0.999988]
       id  Heart Disease
0  630000       0.796671
1  630001       0.065409
2  630002       0.891611
3  630003       0.025243
4  630004       0.443509

✅ submission_multiseed_v3.csv saved!


In [23]:
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'  Seeds             : {CFG.seeds}')
print(f'  Total models      : {len(CFG.seeds) * CFG.n_splits * 3} (3 algos × {CFG.n_splits} folds × {len(CFG.seeds)} seeds)')
print(f'  LightGBM OOF AUC : {oof_auc_lgb:.6f}')
print(f'  XGBoost  OOF AUC : {oof_auc_xgb:.6f}')
print(f'  CatBoost OOF AUC : {oof_auc_cat:.6f}')
print(f'  Ensemble OOF AUC : {best_auc:.6f}')
print(f'  Best weights      : LGB={best_weights[0]:.2f}, XGB={best_weights[1]:.2f}, CAT={best_weights[2]:.2f}')
print('=' * 60)
print('\n→ submission_multiseed_v3.csv を提出してください')

FINAL SUMMARY
  Seeds             : [42, 123, 456, 789, 2024]
  Total models      : 75 (3 algos × 5 folds × 5 seeds)
  LightGBM OOF AUC : 0.955170
  XGBoost  OOF AUC : 0.955195
  CatBoost OOF AUC : 0.955358
  Ensemble OOF AUC : 0.955364
  Best weights      : LGB=0.10, XGB=0.10, CAT=0.80

→ submission_multiseed_v3.csv を提出してください
